In [51]:
import fastf1
import plotly.graph_objects as go
import yaml

with open("C:/Users/boncl/Repos/simply-lovely/config/teams_drivers_info.yaml", "r") as file:
    config = yaml.safe_load(file)
fastf1.Cache.enable_cache('C:/Users/boncl/Repos/simply-lovely/fastf1_cache')

compound_colours = {
    "SOFT": "#E10600",
    "MEDIUM": "#FFD12E",
    "HARD": "#FAF9F6",
    "INTERMEDIATE": "#43B02A",
    "WET": "#0067AD"
}

In [52]:
silverstone_laps = fastf1.get_session(2026, 'Silverstone', 'R')
silverstone_laps.load()
laps = silverstone_laps.laps
laps

core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['16', '63', '44', '1', '6', '30', '41', '5', '43', '10', '81', '87', '31', '11', '12', '77', '55', '14', '18', '3', '23', '27']


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:57:20.252000,NOR,1,0 days 00:01:40.454000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:39.496000,...,True,McLaren,0 days 00:55:39.562000,2026-07-05 14:03:54.055,12,7.0,False,,False,False
1,0 days 00:58:55.088000,NOR,1,0 days 00:01:34.836000,2.0,1.0,NaT,NaT,0 days 00:00:30.170000,0 days 00:00:38.718000,...,True,McLaren,0 days 00:57:20.252000,2026-07-05 14:05:34.745,12,7.0,False,,False,True
2,0 days 01:00:30.329000,NOR,1,0 days 00:01:35.241000,3.0,1.0,NaT,NaT,0 days 00:00:30.061000,0 days 00:00:38.936000,...,True,McLaren,0 days 00:58:55.088000,2026-07-05 14:07:09.581,1,7.0,False,,False,True
3,0 days 01:02:05.445000,NOR,1,0 days 00:01:35.116000,4.0,1.0,NaT,NaT,0 days 00:00:29.899000,0 days 00:00:39.002000,...,True,McLaren,0 days 01:00:30.329000,2026-07-05 14:08:44.822,1,7.0,False,,False,True
4,0 days 01:03:40.599000,NOR,1,0 days 00:01:35.154000,5.0,1.0,NaT,NaT,0 days 00:00:29.884000,0 days 00:00:38.996000,...,True,McLaren,0 days 01:02:05.445000,2026-07-05 14:10:19.938,1,7.0,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1108,0 days 02:15:10.815000,BEA,87,0 days 00:02:23.272000,48.0,3.0,0 days 02:13:11.940000,NaT,0 days 00:00:58.881000,0 days 00:00:51.057000,...,False,Haas F1 Team,0 days 02:12:47.543000,2026-07-05 15:21:02.036,4,14.0,False,,False,False
1109,0 days 02:17:40.547000,BEA,87,0 days 00:02:29.732000,49.0,3.0,NaT,NaT,0 days 00:00:33.070000,0 days 00:01:08.250000,...,False,Haas F1 Team,0 days 02:15:10.815000,2026-07-05 15:23:25.308,4,14.0,False,,False,False
1110,0 days 02:19:23.259000,BEA,87,0 days 00:01:42.712000,50.0,3.0,NaT,NaT,0 days 00:00:37.140000,0 days 00:00:39.158000,...,False,Haas F1 Team,0 days 02:17:40.547000,2026-07-05 15:25:55.040,4,14.0,False,,False,False
1111,0 days 02:20:59.433000,BEA,87,0 days 00:01:36.174000,51.0,3.0,NaT,NaT,0 days 00:00:29.887000,0 days 00:00:39.571000,...,False,Haas F1 Team,0 days 02:19:23.259000,2026-07-05 15:27:37.752,4,14.0,False,,False,False


In [53]:
fig = go.Figure()

for driver in laps["Driver"].unique():
    driver_laps = laps[laps["Driver"] == driver]
    driver_team = laps.loc[laps['Driver']== driver, 'Team'].unique()[0]
    fig.add_scatter(
        x=driver_laps["LapNumber"],
        y=driver_laps["Position"],
        mode="lines",
        name=driver,
        line=dict(
            color=f"rgba({config['teams'][driver_team]['drivers'][driver]['colour']})"
        ))

fig.update_yaxes(autorange="reversed")
fig.show()

In [54]:
fig2 = go.Figure()

driver_order = (
    laps.groupby("Driver")["Position"]
    .last()
    .sort_values()
    .index
    .tolist()
)

for driver in laps["Driver"].unique():
    driver_laps = laps[laps["Driver"] == driver]

    for _, lap in driver_laps.iterrows():
        fig2.add_trace(
            go.Bar(
                x = [1],
                y = [driver],
                base = lap['LapNumber'] -1,
                orientation='h',
                marker_color = compound_colours[lap['Compound']],
                marker_line_width=0,
                showlegend= False
            )
        )

fig2.update_layout(
    barmode="stack",
    xaxis_title="Lap",
    yaxis_title="Driver",
    title = "Tyre strategies over Silverstone GP",
    template="plotly_white",
    bargap=0.2,
    height = 600
)

fig2.update_yaxes(
    categoryorder="array",
    categoryarray=driver_order,
    autorange="reversed"
)
fig2.update_xaxes(showgrid = True, gridwidth = 1, gridcolor = 'lightgrey')

fig2.show()

In [ ]:
#TODO: fix fastest driver lap. mayebt he line could show the average fastest lap, and the fastest lap itself should just be info next to it?

fig3 = go.Figure()
laps_violin = laps.pick_quicklaps()
fastest_lap_overall = laps_violin['LapTime'].dt.total_seconds().min()
fastest_driver = laps_violin.loc[laps_violin['LapTime'] == fastest_lap_overall, 'Driver']
print(fastest_driver)
average_lap_time_overall = laps_violin['LapTime'].dt.total_seconds().mean()

for driver in laps_violin['Driver'].unique():
    driver_laps = laps_violin[laps_violin['Driver'] == driver]
    driver_mean = driver_laps['LapTime'].dt.total_seconds().mean()
    driver_team = driver_laps['Team'].iat[0]
    driver_colour = config['teams'][driver_team]['drivers'][driver]['colour']

    fig3.add_trace(go.Violin(
        y=driver_laps['LapTime'].dt.total_seconds(),
        name= driver,
        showlegend= False,
        line=dict(color=f"rgba({driver_colour})")
    ))
    fig3.add_trace(go.Scatter(
        y = [driver_mean],
        x = [driver],
        mode ="markers",
        marker= dict(symbol="diamond", size = 5, color=f"rgba({driver_colour})"
                ),
        showlegend = False)
    )

fig3.add_hline(y = fastest_lap_overall, line_dash="dash", line_color = "red", annotation_text=f"Fastest Lap by {fastest_driver}: {fastest_lap_overall} s")
fig3.update_yaxes(title="Lap Time in s")
fig3.update_xaxes(title= "Driver")
fig3.update_layout(title = "Spread of Lap Time per driver over race")
fig3.show()

Series([], Name: Driver, dtype: object)
